In [2]:
import pandas as pd


json_path = '/home/antares/Tesis_Data/VicomTech/dmd/gE/30/s3/gE_30_s3_2019-03-15T10;56;02+01;00_rgb_ann_distraction.json'

In [2]:
df =  pd.read_json(json_path)
df

,openlabel
metadata,"{'schema_version': '1.0.0', 'name': 'gE_30_s3_..."
objects,"{'0': {'name': '30', 'type': 'driver', 'frame_..."
frames,"{'0': {'objects': {'0': {}}, 'contexts': {'0':..."
frame_intervals,"[{'frame_start': 0, 'frame_end': 1864}]"
ontologies,{'0': 'http://dmd.vicomtech.org/ontology'}
streams,{'face_camera': {'description': 'Frontal face ...
contexts,"{'0': {'name': '', 'type': 'recording_context'..."
actions,"{'0': {'name': '', 'type': 'gaze_on_road/looki..."


In [3]:
import vcd.core as core

# Create empty vcd object
myVCD = core.VCD()

# Load a VCD file
myVCD.load_from_file(json_path)



In [4]:
myVCD.data['openlabel']['metadata']

{'schema_version': '1.0.0',
 'name': 'gE_30_s3_2019-03-15T10;56;02+01;00_distraction',
 'annotator': '0'}

In [5]:
import pandas as pd
from vcd import core

def extraer_actividades_vcd(file_path):
    # 1. Instanciar y cargar el archivo VCD
    vcd_data = core.VCD()
    vcd_data.load_from_file(file_path)

    # 2. Obtener el nombre del video desde los metadatos
    try:
        video_name = vcd_data.data['openlabel']['metadata']['name']
    except KeyError:
        video_name = file_path

    # 3. Extraer las ACCIONES en lugar de los objetos
    acciones = vcd_data.get_actions()

    filas_datos = []

    # 4. Iterar sobre cada acción y sus intervalos de tiempo
    for act_id, act_data in acciones.items():
        # El tipo de acción (ej: 'drinking', 'talking_on_phone', etc.)
        actividad = act_data.get('type', 'Desconocido')

        # Extraemos la lista de intervalos de frames
        intervalos = act_data.get('frame_intervals', [])

        # Por cada segmento temporal en el que ocurre esta acción, creamos una fila
        for intervalo in intervalos:
            frame_inicio = intervalo.get('frame_start')
            frame_fin = intervalo.get('frame_end')

            filas_datos.append({
                'Video_Origen': video_name,
                'ID_Accion': act_id,
                'Actividad': actividad,
                'Frame_Inicio': frame_inicio,
                'Frame_Fin': frame_fin
            })

    # 5. Convertir a DataFrame
    df_actividades = pd.DataFrame(filas_datos)

    return df_actividades

# --- Ejecución ---
# Sustituye con tu archivo real (el que genera la salida que me mostraste)
archivo_json = "/mnt/c/Users/user/Documents/Alejandro_Mesa_Gomez/Vicomtech/dmd/"

tabla_actividades = extraer_actividades_vcd(archivo_json)
print(tabla_actividades.to_string())

IsADirectoryError: [Errno 21] Is a directory: '/mnt/c/Users/user/Documents/Alejandro_Mesa_Gomez/Vicomtech/dmd/'

In [3]:
!pwd

/mnt/c/Users/user/Documents/Alejandro_Mesa_Gomez/driver_behavior/src/Notebooks


In [8]:
import pandas as pd
from vcd import core
from pathlib import Path
import os

def extraer_solo_acciones_vcd(ruta_archivo):
    """Extrae únicamente las acciones de un archivo VCD."""
    vcd_data = core.VCD()
    
    try:
        # Cargar el archivo VCD
        vcd_data.load_from_file(str(ruta_archivo))
    except Exception as e:
        print(f"Error al leer el archivo {ruta_archivo}: {e}")
        return pd.DataFrame() # Si hay error, devuelve un DataFrame vacío

    # Intentar obtener el nombre del video desde los metadatos
    try:
        video_name = vcd_data.data['openlabel']['metadata']['name']
    except KeyError:
        video_name = os.path.basename(ruta_archivo)

    filas_datos = []

    # Extraer EXCLUSIVAMENTE la sección de acciones
    acciones = vcd_data.get_actions()
    
    if acciones:
        for act_id, act_data in acciones.items():
            # Extraemos la actividad específica
            actividad = act_data.get('type', 'Desconocido')
            intervalos = act_data.get('frame_intervals', [])
            
            # Por cada segmento temporal de la acción, creamos una fila
            for intervalo in intervalos:
                filas_datos.append({
                    'Ruta_Archivo': str(ruta_archivo),
                    'Video_Origen': video_name,
                    'ID_Accion': act_id,
                    'Actividad': actividad,
                    'Frame_Inicio': intervalo.get('frame_start'),
                    'Frame_Fin': intervalo.get('frame_end')
                })

    return pd.DataFrame(filas_datos)

def procesar_directorio_acciones(directorio_raiz):
    """Busca JSONs en carpetas anidadas y extrae solo sus acciones."""
    rutas_json = Path(directorio_raiz).rglob('*.json')
    
    lista_dataframes = []
    archivos_procesados = 0

    print(f"Iniciando búsqueda de acciones en: {directorio_raiz}")
    
    for ruta in rutas_json:
        df_temporal = extraer_solo_acciones_vcd(ruta)
        
        if not df_temporal.empty:
            lista_dataframes.append(df_temporal)
            archivos_procesados += 1
            
    print(f"Se procesaron {archivos_procesados} archivos con acciones exitosamente.")

    # Unificar todo en una sola tabla
    if lista_dataframes:
        df_final = pd.concat(lista_dataframes, ignore_index=True)
        return df_final
    else:
        print("No se encontraron acciones en los archivos procesados.")
        return pd.DataFrame()

# --- Ejecución del script ---
# Cambia esta ruta por la de tu carpeta con todos los subdirectorios
carpeta_principal = "/mnt/c/Users/user/Documents/Alejandro_Mesa_Gomez/Vicomtech/dmd/"

tabla_maestra = procesar_directorio_acciones(carpeta_principal)

if not tabla_maestra.empty:
    # Mostramos las primeras filas en consola
    print(tabla_maestra.head().to_string())
    
    # Exportamos el resultado a un CSV limpio, sin objetos
    tabla_maestra.to_csv("anotaciones_completas.csv", index=False)
    print("\n¡Archivo 'solo_acciones_extraidas.csv' guardado con éxito!")

#

Iniciando búsqueda de acciones en: /mnt/c/Users/user/Documents/Alejandro_Mesa_Gomez/Vicomtech/dmd/
Se procesaron 169 archivos con acciones exitosamente.
                                                                                                                        Ruta_Archivo                                   Video_Origen ID_Accion                  Actividad  Frame_Inicio  Frame_Fin
0  /mnt/c/Users/user/Documents/Alejandro_Mesa_Gomez/Vicomtech/dmd/gA/1/s1/gA_1_s1_2019-03-08T09;31;15+01;00_rgb_ann_distraction.json  gA_1_s1_2019-03-08T09;31;15+01;00_distraction         0  gaze_on_road/looking_road             0         33
1  /mnt/c/Users/user/Documents/Alejandro_Mesa_Gomez/Vicomtech/dmd/gA/1/s1/gA_1_s1_2019-03-08T09;31;15+01;00_rgb_ann_distraction.json  gA_1_s1_2019-03-08T09;31;15+01;00_distraction         0  gaze_on_road/looking_road            44        516
2  /mnt/c/Users/user/Documents/Alejandro_Mesa_Gomez/Vicomtech/dmd/gA/1/s1/gA_1_s1_2019-03-08T09;31;15+01;00_rgb_ann_distr